# 01 - IRSA query: real SPHEREx data spike (Task 3)
Goal: query IRSA for one sky position, download real cutouts, inspect every extension, read the **wavelength at the target pixel**, and export a PNG + metadata for the frontend.
Based on IRSA's tutorial: https://caltech-ipac.github.io/irsa-tutorials/spherex-cutouts/

**Differences from the written steps (and why):**
1. Two lines in the written steps were corrupted (`image.shape [..]`, `results["uri"] [..]`); they should be `image.shape[1]` and `results["uri"][1]`.
2. `norm="log"` fails on negative or NaN pixels; we use a robust percentile stretch.
3. Step 3.15 subtracted the first two results. Those are usually at **different wavelengths** (wavelength varies across a SPHEREx detector), so that difference mostly shows wavelength, not change. Here we read the wavelength at the target pixel and only compare frames that match.
4. Added: wavelength at the target pixel (from WCS-WAVE), which the whole app depends on.

In [ ]:
!pip install numpy astropy astroquery matplotlib

In [ ]:
import json, os, time, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.ipac.irsa import Irsa
from astropy.utils.data import conf
conf.remote_timeout = 120

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC, REP = ROOT/"data/raw", ROOT/"data/processed", ROOT/"data/reports"
for d in (RAW, PROC, REP): d.mkdir(parents=True, exist_ok=True)

## 3.2 Target

In [ ]:
target_name = "Barnard's Star"        # or: "WISE 0855"  (ra=133.795, dec=-7.245)
coord = SkyCoord(ra=269.452*u.deg, dec=4.693*u.deg, frame="icrs")
size = 0.1*u.deg                       # cutout width
bandpass = "SPHEREx-D2"                # one detector; SPHEREx-D1 .. SPHEREx-D6
print(target_name, coord, size, bandpass)

## 3.3 Query IRSA (TAP)
Every SPHEREx spectral image whose footprint contains this position, for this detector, oldest first.

In [ ]:
query = f"""
SELECT 'https://irsa.ipac.caltech.edu/' || a.uri AS uri,
       p.time_bounds_lower, p.energy_bandpassname
FROM spherex.artifact a
JOIN spherex.plane p ON a.planeid = p.planeid
WHERE 1 = CONTAINS(POINT('ICRS', {coord.ra.deg}, {coord.dec.deg}), p.poly)
  AND p.energy_bandpassname = '{bandpass}'
ORDER BY p.time_bounds_lower
"""
from astropy.table import Table

def run_tap(query, tries=5):
    """IRSA's TAP answers 502/503 when overloaded. Retry with backoff, alternating sync and async jobs."""
    last = None
    for k in range(tries):
        try:
            res = Irsa.query_tap(query, async_job=(k % 2 == 1))
            return res.to_table() if hasattr(res, "to_table") else res
        except Exception as e:
            last = e
            wait = 10 * 2**k
            print(f"attempt {k+1}/{tries} failed ({'async' if k % 2 else 'sync'}): {repr(e)[:160]}\n  retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"TAP query failed after {tries} tries: {repr(last)[:200]}")

saved = REP/"irsa_observations.csv"
try:
    results = run_tap(query)
    results.write(saved, overwrite=True)            # keep a copy so an IRSA outage cannot block you later
except RuntimeError:
    if saved.exists():
        results = Table.read(saved)
        print("IRSA unreachable: USING THE SAVED LIST from", saved)
    else:
        raise
print("Number of images found:", len(results))
results[:5]

In [ ]:
# SANITY: what unit is time_bounds_lower? The backend assumes MJD (about 60000-62000 for 2023-2028).
t0 = float(results["time_bounds_lower"][0]); print("first time value:", t0)
print("looks like MJD" if 55000 < t0 < 70000 else "NOT MJD: tell the backend owner, app/services/irsa.py assumes MJD")
print("all rows .fits?", all(str(u_).split("?")[0].endswith(".fits") for u_ in results["uri"]))

## 3.4 Save the observation list

In [ ]:
results.write(REP/"irsa_observations.csv", overwrite=True)
print("Saved", REP/"irsa_observations.csv")

## 3.5 Cutout URL
`<uri>?center=<RA>,<Dec>d&size=<deg>` : `uri` is the full SPHEREx FITS file at IRSA; `center` is the cutout centre (note the **d** after Dec); `size` is the width in degrees.

In [ ]:
import urllib.error

UA = {"User-Agent": "SkyDetective/0.1 (NASA Space Apps 2026 project)"}

def cutout_url(uri, coord, size):
    return f"{uri}?center={coord.ra.deg},{coord.dec.deg}d&size={size.to_value(u.deg)}"

def download(url, path, tries=5):
    """Retry with backoff: IRSA's cutout service sometimes answers 503 when busy. Checks the bytes are really FITS."""
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    last = None
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=180) as r:
                data = r.read()
            if not data.startswith(b"SIMPLE"):
                raise ValueError("not a FITS file, got: " + data[:100].decode(errors="replace"))
            path.write_bytes(data)
            return path
        except Exception as e:
            last = e
            body = e.read()[:300].decode(errors="replace") if isinstance(e, urllib.error.HTTPError) else ""
            wait = 5 * 2**k
            print(f"attempt {k+1}/{tries} failed: {e!r} {body}\n  retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"download failed after {tries} tries: {last!r}")

url0 = cutout_url(results["uri"][0], coord, size)
print(url0)
p0 = download(url0, RAW/"test_cutout.fits")
print("Downloaded", p0, f"{p0.stat().st_size/1e6:.2f} MB")

### If it still fails with 503: find out which of the three causes it is
1. **IRSA is busy or down** (most common): the example URL below also fails, and so does another row. Wait 10-15 minutes and retry.
2. **This one file is a problem**: the example works but row 0 fails. Try the next rows.
3. **Your network blocks it** (Colab shares IPs, which IRSA can rate-limit): it works in your own browser or terminal but not here.

In [ ]:
tests = {
    "row 0, no cutout (HEAD)": ("HEAD", str(results["uri"][0])),
    "row 1 cutout": ("GET", cutout_url(results["uri"][1], coord, size)),
    "row 5 cutout": ("GET", cutout_url(results["uri"][min(5, len(results)-1)], coord, size)),
}
for name, (method, url) in tests.items():
    try:
        req = urllib.request.Request(url, headers=UA, method=method)
        with urllib.request.urlopen(req, timeout=60) as r:
            print(f"{name}: HTTP {r.status}")
    except Exception as e:
        print(f"{name}: {e!r}")

## 3.7 Every extension

In [ ]:
hdul = fits.open(p0)
hdul.info()

## 3.8 Image

In [ ]:
image = hdul["IMAGE"].data
print("shape", image.shape, "dtype", image.dtype, "BUNIT", hdul["IMAGE"].header.get("BUNIT"))
print("min/median/max", np.nanmin(image), np.nanmedian(image), np.nanmax(image), "| NaN pixels:", int(np.isnan(image).sum()))

def stretch(a, lo=1, hi=99):
    v = np.nanpercentile(a, [lo, hi]); return np.clip((a - v[0]) / (v[1] - v[0] + 1e-30), 0, 1)

plt.figure(figsize=(7,7)); plt.imshow(stretch(image), origin="lower", cmap="inferno")
plt.title(f"{target_name}  {bandpass}"); plt.xlabel("x pixel"); plt.ylabel("y pixel"); plt.colorbar(label="stretched"); plt.show()

## 3.9 WCS: where is the target on this image?

In [ ]:
hdr = hdul["IMAGE"].header
wcs = WCS(hdr)                      # celestial WCS of the cutout
sky = wcs.celestial
tx, ty = sky.world_to_pixel(coord)
print("target pixel (x, y):", float(tx), float(ty), "| image centre:", image.shape[1]/2, image.shape[0]/2)
from astropy.wcs.utils import proj_plane_pixel_scales
print("pixel scale (arcsec/px):", proj_plane_pixel_scales(sky) * 3600)

## 3.10 Quality, uncertainty, background

In [ ]:
flags = hdul["FLAGS"].data; var = hdul["VARIANCE"].data; zodi = hdul["ZODI"].data
print("FLAGS  unique values:", np.unique(flags)[:20], "| flagged fraction:", float((flags != 0).mean()))
print("  set bits:", [b for b in range(32) if np.any((flags.astype('int64') >> b) & 1)])
print("VARIANCE  min/max:", np.nanmin(var), np.nanmax(var))
print("ZODI  median:", np.nanmedian(zodi))
print("extension names:", [h.name for h in hdul])

## 3.11 Header metadata (which header holds which key)

In [ ]:
for key in ["DATE-OBS", "MJD-OBS", "MJD-AVG", "EXPTIME", "DETECTOR", "VERSION", "BUNIT"]:
    print(f"{key:9s} primary={hdul[0].header.get(key)!r:28} image={hdul['IMAGE'].header.get(key)!r}")

## NEW: wavelength at the target pixel (WCS-WAVE)
A SPHEREx detector has a different wavelength at every pixel, so the same sky position lands on a **different wavelength in each exposure**. The backend groups frames by this value, so confirm it reads correctly here first.

In [ ]:
def wavelength_at(hdul, x, y):
    w = WCS(hdul["IMAGE"].header, fobj=hdul, key="W")     # spectral WCS stored via the WCS-WAVE extension
    out = w.pixel_to_world(x, y)
    wave, band = (out[0], out[1]) if isinstance(out, (tuple, list)) else (out, None)
    return float(wave.to_value(u.um)), (float(band.to_value(u.um)) if band is not None else None)

wl, bw = wavelength_at(hdul, float(tx), float(ty))
print(f"wavelength at target: {wl:.4f} um   bandpass width: {bw}")
print("at image corners:", [round(wavelength_at(hdul, x, y)[0], 3) for x, y in [(0,0), (image.shape[1]-1, 0), (0, image.shape[0]-1)]])

## 3.12 Browser-ready PNG (north up, east left, origin bottom-left, like the frontend expects)

In [ ]:
import matplotlib.cm as cm
rgb = (cm.inferno(stretch(image))[:, :, :3] * 255).astype(np.uint8)
png0 = PROC/"test_cutout.png"
plt.imsave(png0, rgb, origin="lower"); print("Saved", png0)

## 3.13 Several observations + their wavelengths at the target
Download the first N, read the wavelength at the target pixel in each, and group by wavelength (2% tolerance, same as the backend and frontend).

In [ ]:
N = 12
rows = []
for i in range(min(N, len(results))):
    try:
        pth = download(cutout_url(results["uri"][i], coord, size), RAW/f"cutout_{i:02d}.fits")
        with fits.open(pth) as h:
            x, y = WCS(h["IMAGE"].header).celestial.world_to_pixel(coord)
            wl, _ = wavelength_at(h, float(x), float(y))
            rows.append(dict(i=i, path=str(pth), time=float(results["time_bounds_lower"][i]), wavelength_um=round(wl, 4),
                             on_image=bool(0 <= x < h["IMAGE"].data.shape[1] and 0 <= y < h["IMAGE"].data.shape[0])))
    except Exception as e:
        print("skip", i, repr(e)[:120])
import pandas as pd
df = pd.DataFrame(rows); df

In [ ]:
groups, tol = [], 0.02
for r in sorted(rows, key=lambda r: r["wavelength_um"]):
    if groups and (r["wavelength_um"] - groups[-1][0]["wavelength_um"]) / groups[-1][0]["wavelength_um"] <= tol: groups[-1].append(r)
    else: groups.append([r])
for g in groups: print(f"{g[0]['wavelength_um']:.3f} um: {len(g)} frame(s)", [r['i'] for r in g])
print("\nIf every group has ONE frame, you will not get a same-wavelength difference from this detector in the first N rows.",
      "SPHEREx revisits a given wavelength about every 6 months; widen N or query the full list.")

## 3.14 / 3.15 Blink and difference (only for frames at a matching wavelength)
If no group has two frames yet, the blink below is still useful to see the alignment, but **do not subtract**: it would be a wavelength difference.

In [ ]:
pair = next((g[:2] for g in groups if len(g) >= 2), None)
if pair is None:
    print("No same-wavelength pair yet. Increase N, or try another target/detector.")
else:
    pair = sorted(pair, key=lambda r: r["time"])
    a, b = (fits.getdata(r["path"], "IMAGE") for r in pair)
    print("pair:", [(r["i"], r["wavelength_um"], r["time"]) for r in pair], "| shapes", a.shape, b.shape)
    a = a - np.nanmedian(a); b = b - np.nanmedian(b)          # per-frame background
    if a.shape == b.shape:
        d = b - a                                              # later minus earlier
        s = 5 * 1.4826 * np.nanmedian(np.abs(d - np.nanmedian(d)))
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(stretch(a), origin="lower", cmap="inferno"); ax[0].set_title("earlier")
        ax[1].imshow(stretch(b), origin="lower", cmap="inferno"); ax[1].set_title("later")
        im = ax[2].imshow(d, origin="lower", cmap="coolwarm", vmin=-s, vmax=s); ax[2].set_title("later - earlier (red = brighter now)")
        plt.show()
        print("CAUTION: these cutouts share a shape but are NOT yet on a common WCS grid. Stars leaving +/- pairs mean alignment is needed (reproject), which the backend pipeline does.")

## Save a metadata JSON for the frontend / backend hand-off

In [ ]:
meta = dict(target=target_name, ra=coord.ra.deg, dec=coord.dec.deg, cutout_size_deg=size.to_value(u.deg), bandpass=bandpass,
            n_observations=len(results), frames=rows,
            extensions=[h.name for h in hdul], image_shape=list(image.shape),
            png="data/processed/test_cutout.png")
(REP/"cutout_metadata.json").write_text(json.dumps(meta, indent=2)); print("Saved", REP/"cutout_metadata.json")

## 3.16 Findings (fill in; also copy to `research/03-data-schema.md`)
- Query worked: ___ ; observations (D2): ___ ; time range: ___ ; time_bounds_lower is MJD: Y/N
- Cutout download worked: ___ ; size: ___ MB ; image shape: ___ ; BUNIT: ___ ; pixel scale: ___ "/px
- Extensions present: ___ (expected IMAGE, FLAGS, VARIANCE, ZODI, EPSF, WCS-WAVE)
- FLAGS bits seen: ___ ; fraction flagged: ___
- Wavelength at target readable (WCS-WAVE, key W): Y/N ; range across the image: ___ um
- Header keys: which of DETECTOR / MJD-AVG / EXPTIME are in the primary vs IMAGE header: ___
- Same-wavelength pairs available in the first N rows: ___
- Barnard's Star moves about 10 arcsec/yr, i.e. ~1.7 px/yr at 6 arcsec per pixel: expect to see <1 px over 6 months. Prepare a fallback target.